# Compare This Repository With the Official Paper Code

This notebook is intentionally light. It does not merge the two pipelines into one wrapper. Instead, it prepares both codebases in the same Colab runtime so you can run this repository and the official notebook-based implementation under comparable seeds and compute budgets.

## 1. Clone This Repository With the Official Submodule

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/FrancescoVinciguerra/rea_of_llms_supplementary_code-main"
REPO_DIR = Path("/content/rea_of_llms_supplementary_code-main")

if REPO_DIR.exists():
    print(f"Repository already exists: {REPO_DIR}")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "submodule", "update", "--init", "--recursive"], check=True)
else:
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, str(REPO_DIR)], check=True)

OFFICIAL_DIR = REPO_DIR / "external" / "rea_of_llms_official"
print("Local repo:", REPO_DIR)
print("Official paper code:", OFFICIAL_DIR)

## 2. Install Shared Dependencies

The official repository uses notebooks and its own `pyproject.toml`. In Colab, this notebook installs the dependency set from this repository to avoid forcing the official CPU-only PyTorch index. The required packages for the official notebooks are covered by the shared set.

In [ ]:
%cd /content/rea_of_llms_supplementary_code-main
!pip install -r requirements.txt

## 3. Shared Comparison Settings

The official `sampling_examples.ipynb` small example uses `TORCH_SEED = 42`, `PY_SEED = 1234`, `MAX_GEN_TOKENS = 100`, `steps_per_bias = 10`, and `num_trajectories = 3`. The SMC command below uses the same prompt, model, tokenizer, seed, completion length, and comparable ARI bias values. Compute is not exactly identical because TPS edits suffixes while this SMC implementation proposes full completions.

In [ ]:
MODEL_NAME = "roneneldan/TinyStories-8M"
TOKENIZER_NAME = "EleutherAI/gpt-neo-125M"
PROMPT = "Once upon a time, in a big forest, there lived a rhinoc"
SEED = 42
MAX_NEW_TOKENS = 100
SMC_PARTICLES = 3
SMC_MCMC_STEPS_PER_LEVEL = 10
SMC_LAMBDAS = "0,-0.1,-0.2,-0.3"

smc_generated_token_budget = SMC_PARTICLES * MAX_NEW_TOKENS * (1 + (len(SMC_LAMBDAS.split(',')) - 1) * SMC_MCMC_STEPS_PER_LEVEL)
print("Approximate SMC generated completion-token transitions:", smc_generated_token_budget)

## 4. Run This Repository's Full-Sequence SMC

This writes outputs under `outputs/comparison_smc_seed42/` and selected summaries under `results/selected_runs/`.

In [ ]:
!python full_sequence_smc_experiment/run_smc_sampler.py \
  --model-name "$MODEL_NAME" \
  --tokenizer-name "$TOKENIZER_NAME" \
  --prompt "$PROMPT" \
  --device auto \
  --seed "$SEED" \
  --num-particles "$SMC_PARTICLES" \
  --max-new-tokens "$MAX_NEW_TOKENS" \
  --lambdas "$SMC_LAMBDAS" \
  --mcmc-steps-per-level "$SMC_MCMC_STEPS_PER_LEVEL" \
  --run-name comparison_smc_seed42 \
  --experiment-log results/EXPERIMENT_LOG.md \
  --publish-result-dir results/selected_runs \
  --output-dir outputs/comparison_smc_seed42

In [ ]:
from pathlib import Path

report_path = Path("outputs/comparison_smc_seed42/report.md")
summary_path = Path("outputs/comparison_smc_seed42/summary.json")
print("SMC summary:", summary_path)
print(report_path.read_text(encoding="utf-8"))

## 5. Run the Official Paper Code Separately

The official repository is included as a Git submodule at `external/rea_of_llms_official`. Its pipeline is notebook-based:

- `sampling_examples.ipynb`: direct sampling and annealed TPS examples;
- `reweighting_example.ipynb`: burn-in, Gelman-Rubin, MBAR, bootstrap intervals.

Run the cells below only when you want to execute the official notebooks in this same Colab runtime. Keep `RUN_OFFICIAL_SAMPLING_NOTEBOOK = False` until you are ready.

In [ ]:
from IPython.display import display, Markdown

display(Markdown(f"Official code directory: `{OFFICIAL_DIR}`"))
display(Markdown(f"Official sampling notebook: `{OFFICIAL_DIR / 'sampling_examples.ipynb'}`"))
display(Markdown(f"Official reweighting notebook: `{OFFICIAL_DIR / 'reweighting_example.ipynb'}`"))

print("Official notebook files:")
!ls -lh external/rea_of_llms_official/*.ipynb

In [ ]:
RUN_OFFICIAL_SAMPLING_NOTEBOOK = False

if RUN_OFFICIAL_SAMPLING_NOTEBOOK:
    ip = get_ipython()
    ip.run_line_magic("cd", "/content/rea_of_llms_supplementary_code-main/external/rea_of_llms_official")
    ip.run_line_magic("run", "sampling_examples.ipynb")
else:
    print("Set RUN_OFFICIAL_SAMPLING_NOTEBOOK = True to execute the official sampling notebook.")

In [ ]:
RUN_OFFICIAL_REWEIGHTING_NOTEBOOK = False

if RUN_OFFICIAL_REWEIGHTING_NOTEBOOK:
    ip = get_ipython()
    ip.run_line_magic("cd", "/content/rea_of_llms_supplementary_code-main/external/rea_of_llms_official")
    ip.run_line_magic("run", "reweighting_example.ipynb")
else:
    print("After running the official sampling notebook, set RUN_OFFICIAL_REWEIGHTING_NOTEBOOK = True to execute reweighting/MBAR.")

## 6. Inspect Official Outputs

The official small example writes JSON files under `external/rea_of_llms_official/data/example/`. This cell prints compact metadata if those files exist.

In [ ]:
import json

example_dir = OFFICIAL_DIR / "data" / "example"
print("Official example output directory:", example_dir)
if not example_dir.exists():
    print("No official outputs yet. Run the official sampling notebook first.")
else:
    for path in sorted(example_dir.glob("*.json")):
        with path.open("r", encoding="utf-8") as handle:
            data = json.load(handle)
        if isinstance(data, list):
            print(path.name, "top-level length:", len(data))
        else:
            print(path.name, type(data).__name__)